# Membuat Embedding Dokumen (Google Colab)
Notebook ini membuat `artifacts/embeddings.npy` dari `documents_processed.csv` (hasil `scripts.build_dataset`) memakai model **intfloat/multilingual-e5-small**.
- Tanpa fine-tuning, tanpa API eksternal.
- Embedding dibuat **sekali** di sini; aplikasi Streamlit hanya membacanya.

**Persiapan:** unggah seluruh folder proyek `capstone-semantic-retrieval` ke Google Drive (termasuk folder `src/` dan folder artifacts/artifacts_real yang berisi `documents_processed.csv`).

## 1. Install library

In [ ]:
# Install library yang dibutuhkan (jalankan sekali per sesi Colab)
!pip install -q sentence-transformers pandas numpy

## 2. Mount Google Drive dan atur PROJECT_DIR

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# >>> UBAH jika folder proyek Anda berada di lokasi lain <<<
PROJECT_DIR = '/content/drive/MyDrive/capstone-semantic-retrieval'

In [ ]:
import os, sys, json
sys.path.insert(0, PROJECT_DIR)  # agar folder src/ bisa di-import

# Pilih dataset. 'synthetic' = data contoh (folder artifacts); 'real' = paper nyata (folder artifacts_real).
# Keduanya dibuat lebih dulu di komputer lokal dengan:  python -m scripts.build_dataset --kind real|synthetic
DATASET = 'synthetic'   # ganti ke 'real' untuk dataset paper nyata
ARTIFACTS_DIR = os.path.join(PROJECT_DIR, 'artifacts_real' if DATASET == 'real' else 'artifacts')
PROCESSED_PATH = os.path.join(ARTIFACTS_DIR, 'documents_processed.csv')
EMBEDDINGS_PATH = os.path.join(ARTIFACTS_DIR, 'embeddings.npy')
CONFIG_PATH = os.path.join(ARTIFACTS_DIR, 'model_config.json')

assert os.path.exists(PROCESSED_PATH), (
    f'File tidak ditemukan: {PROCESSED_PATH}. Jalankan scripts.build_dataset --kind {DATASET} di komputer lokal '
    'lalu unggah folder hasilnya, atau periksa PROJECT_DIR.')


## 3. Mapping dan pembersihan dataset

In [ ]:
import pandas as pd

# Baca dataset yang SUDAH divalidasi & dideduplikasi oleh scripts.build_dataset (tidak diproses ulang di sini)
df = pd.read_csv(PROCESSED_PATH, encoding='utf-8', dtype=str, keep_default_na=False)
print('Dataset:', DATASET, '| jumlah dokumen:', len(df), '| data_source:', df['data_source'].unique().tolist())
df[['doc_id', 'title', 'year']].head(3)


In [ ]:
# Tampilkan 3 contoh dokumen
for _, row in df.head(3).iterrows():
    print(row['doc_id'], '|', row['title'])
    print('   ', row['document_text'][:200], '...\n')

## 4. Muat model dan cek GPU

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'intfloat/multilingual-e5-small'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Perangkat yang dipakai:', device)  # CPU tetap bisa, hanya lebih lambat
model = SentenceTransformer(MODEL_NAME, device=device)

## 5. Buat embedding dokumen (batch)
Input dokumen **wajib** diawali `passage: `.

In [ ]:
from src.embedding import encode_documents

# Fungsi yang sama dengan scripts/generate_embeddings.py: prefix 'passage: ', ternormalisasi, float32
embeddings = encode_documents(model, df, batch_size=32, show_progress=True)

## 6. Simpan embedding dan konfigurasi model

In [ ]:
from src.embedding import library_versions, write_embedding_artifacts

# Memvalidasi (jumlah, normalisasi, NaN) lalu menulis embeddings.npy + model_config.json (termasuk hash doc_id)
write_embedding_artifacts(ARTIFACTS_DIR, df, embeddings, MODEL_NAME, extra={'library_versions': library_versions()})

print('Jumlah dokumen valid :', len(df))
print('Shape embedding      :', embeddings.shape)
print('Ukuran file embedding: %.2f MB' % (os.path.getsize(EMBEDDINGS_PATH) / 1024 / 1024))
print('Berikutnya, di komputer lokal: python -m scripts.check_embeddings --artifacts-dir <folder> --sanity')

## 7. Uji semantic search untuk satu kueri
Kueri diawali `query: `. Ini hanya pengujian; aplikasi tidak membuat ulang embedding dokumen.

In [ ]:
kueri_pengguna = 'banjir'   # ganti sesuai kebutuhan

q_vec = model.encode(['query: ' + kueri_pengguna], normalize_embeddings=True)[0]
scores = embeddings @ q_vec
top_idx = np.argsort(-scores)[:5]
for rank, i in enumerate(top_idx, start=1):
    print(f"{rank}. [{scores[i]:.4f}] {df.iloc[i]['doc_id']} - {df.iloc[i]['title']}")

## 8. Mengunduh artifacts ke laptop lokal
1. Buka Google Drive di browser, masuk ke folder `capstone-semantic-retrieval/artifacts/`.
2. Unduh tiga file: `documents_processed.csv`, `embeddings.npy`, `model_config.json`
   (klik kanan -> **Download**).
3. Salin ketiganya ke folder `artifacts/` pada proyek di laptop (timpa file lama).
4. Pastikan jumlah baris `documents_processed.csv` sama dengan jumlah embedding. Jika dataset diubah, jalankan ulang notebook ini.
5. Jalankan aplikasi: `streamlit run app.py`.

Catatan: model akan diunduh otomatis saat aplikasi pertama kali dipakai untuk pencarian semantic (butuh internet sekali). Lakukan sebelum hari demo agar tersimpan di cache lokal.